# 06 — Pipeline Supervisionado

Pré-processamento + partição persistida. Consome o artefato do Notebook 03;
não cria partição e mantém o teste fechado.

In [1]:
from pathlib import Path
import json
import os
import sys


def encontrar_raiz_projeto(start: Path | None = None) -> Path:
    atual = (start or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "src").is_dir() and (candidato / "pyproject.toml").is_file():
            return candidato
    raise RuntimeError(f"Raiz do projeto não encontrada a partir de: {atual}")


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import joblib
import pandas as pd
from scipy import sparse
from sklearn.base import clone

from src import config as C
from src import config as env
from src.registro import Monitor
from src.registro import Run
from src.dados import load_supervised_artifact, split_partitions
from src.avaliacao import check_leakage
from src.modelos import build_pipeline, filtrar_rotulos_validos

env.carregar_dotenv()
env.garantir_diretorios()

print(C.resumo())

ROOT_DIR          = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador
DATA_DIR          = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data
PROCESSED_DIR     = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\processed
ANALYTICAL_DIR    = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\analytical
OUTPUTS_DIR       = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs
RUNS_DIR          = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\runs
CONFIGS_DIR       = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\configs
SUPERVISED_YAML   = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\configs\supervised.yaml (ok)
PARQUET_NAME      = srag.parquet
DATA_VERSION      = dados-v2
GITHUB_REPO       = Ivan-S-Oliveira/Projeto_Integrador
HTTP_TIMEOUT      = 180
HTTP_TENTATIVAS   = 3
SEED              = 42
GITHUB_TOKEN?     = False
SRAG_PARQUET_URL  = https://gi

In [2]:
EXPECTED_KIND = "final"
os.environ["SRAG_COHORT_KIND"] = EXPECTED_KIND

try:
    df, manifest = load_supervised_artifact(
        ROOT_DIR,
        expected_kind=EXPECTED_KIND,
    )
except FileNotFoundError as exc:
    nomes = {"dataset_supervisionado.parquet", "manifesto_supervisionado.json"}
    encontrados = [
        str(p)
        for p in ROOT_DIR.rglob("*")
        if p.is_file() and p.name in nomes
    ]

    detalhe = (
        f"Arquivos encontrados: {encontrados}"
        if encontrados
        else f"Nenhum dos arquivos esperados foi encontrado em {ROOT_DIR}."
    )
    encontrados_nomes = {Path(p).name for p in encontrados}
    ausentes = sorted(nomes - encontrados_nomes)

    raise FileNotFoundError(
        "O artefato supervisionado está incompleto. "
        f"Arquivos ausentes: {ausentes}. "
        "Execute o Notebook 03 adaptado para gerar o "
        "dataset_supervisionado.parquet e o manifesto correspondente. "
        f"{detalhe}"
    ) from exc
treino, validacao = split_partitions(df, manifest)

PARTICAO = manifest["partition_column"]
TARGET   = manifest.get("target_column") or C.cfg("target", "coluna")
FEATURES = list(manifest.get("feature_columns") or C.features_finais())
PROIBIDAS = manifest.get("forbidden_columns") or list(C.cfg("features", "proibidas") or [])

N_TREINO = int(len(treino))
N_VAL    = int(len(validacao))
N_TESTE  = int(sum(int(v) for k, v in manifest["partition_counts"].items() if str(k) in {"teste", "test_ref", "test_pandemic"}))

assert set(df[PARTICAO].astype(str).unique()).isdisjoint({"teste", "test_ref", "test_pandemic"})
assert set(FEATURES).issubset(df.columns)

print(f"treino    : {N_TREINO:,}")
print(f"validacao : {N_VAL:,}")
print(f"teste     : {N_TESTE:,} (não carregado)")
print(f"features  : {len(FEATURES)}")
print(f"proibidas : {len(PROIBIDAS)}")

treino    : 556,462
validacao : 119,243
teste     : 3,035,074 (não carregado)
features  : 30
proibidas : 34


In [3]:
X_train, y_train = filtrar_rotulos_validos(
    treino[FEATURES],
    pd.to_numeric(treino[TARGET], errors="coerce").astype("float64"),
)
X_val, y_val = filtrar_rotulos_validos(
    validacao[FEATURES],
    pd.to_numeric(validacao[TARGET], errors="coerce").astype("float64"),
)

check_leakage(X_train, forbidden_columns=PROIBIDAS)
check_leakage(X_val,   forbidden_columns=PROIBIDAS)

print(f"treino    : {len(X_train):,}  positivos={int(y_train.sum()):,}")
print(f"validacao : {len(X_val):,}  positivos={int(y_val.sum()):,}")

treino    : 556,462  positivos=46,961
validacao : 119,243  positivos=10,064


In [4]:
pipeline = build_pipeline(X=X_train, y=y_train)
pipeline_arquitetura = clone(pipeline)

monitor = Monitor(notebook="C_11_pipeline_supervisionado")

with monitor.etapa("fit_treino"):
    pipeline.fit(X_train, y_train)

with monitor.etapa("transform_val_10k"):
    X_val_proc_sample = pipeline.transform(X_val.head(10_000))

assert sparse.issparse(X_val_proc_sample), (
    f"pré-processador deveria ser esparso; veio {type(X_val_proc_sample).__name__}"
)
assert X_val_proc_sample.dtype.name == "float32", (
    f"pré-processador deveria produzir float32; veio {X_val_proc_sample.dtype}"
)

N_FEATURES_PROCESSADAS = int(X_val_proc_sample.shape[1])

print(f"features : {X_train.shape[1]} -> {N_FEATURES_PROCESSADAS}")
print(
    f"amostra  : {X_val_proc_sample.shape}  "
    f"nnz={sparse.csr_matrix(X_val_proc_sample).nnz:,}"
)
print(f"monitor  : {monitor.salvar()}")

monitor.resumo()

[C_11_pipeline_supervisionado] fit_treino: 5.064s | pico 1320.9 MB | ok
[C_11_pipeline_supervisionado] transform_val_10k: 0.202s | pico 919.7 MB | ok
features : 30 -> 122
amostra  : (10000, 122)  nnz=298,262
monitor  : C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\performance\C_11_pipeline_supervisionado.csv


,notebook,etapa,status,duracao_s,rss_inicio_mb,rss_fim_mb,rss_pico_mb,arrow_inicio_mb,arrow_fim_mb
0,C_11_pipeline_supervisionado,fit_treino,ok,5.064,913.4,914.6,1320.9,118.4,118.4
1,C_11_pipeline_supervisionado,transform_val_10k,ok,0.202,914.6,919.7,919.7,118.4,118.4


In [5]:
import cloudpickle

OUT_DIR = C.OUTPUTS_DIR / "06_pipeline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pipeline_path     = OUT_DIR / "preprocessor.joblib"
pipeline_fit_path = OUT_DIR / "preprocessor_ajustado.joblib"
split_info_path   = OUT_DIR / "split_info.json"

split_info = {
    "origem":            "artefato_supervisionado",
    "cohort_kind":       manifest.get("cohort_kind"),
    "partition_column":  PARTICAO,
    "n_total":           int(manifest["rows"]),
    "n_treino_split":    N_TREINO,
    "n_val_split":       N_VAL,
    "n_teste_split":     N_TESTE,
    "n_treino_rotulado": int(len(y_train)),
    "n_val_rotulado":    int(len(y_val)),
}
split_info_path.write_text(
    json.dumps(split_info, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Arquitetura : {pipeline_path}")
print(f"Ajustado    : {pipeline_fit_path}")
print(f"Split info  : {split_info_path}")

Arquitetura : C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\06_pipeline\preprocessor.joblib
Ajustado    : C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\06_pipeline\preprocessor_ajustado.joblib
Split info  : C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\06_pipeline\split_info.json


In [6]:
pre          = pipeline.named_steps["pre"]
features_num = list(pre.transformers_[0][2])
features_cat = list(pre.transformers_[1][2])

with Run(modelo="06_pipeline_supervisionado", notas="Arquitetura do pipeline supervisionado") as r:
    r.add_dataset(
        n_rows=int(manifest["rows"]),
        train_rows=int(len(y_train)),
        valid_rows=int(len(y_val)),
        test_rows=N_TESTE,
    )
    r.features = {
        "n_features":  len(features_num) + len(features_cat),
        "numericas":   features_num,
        "categoricas": features_cat,
    }
    r.parametros = {
        "target":           TARGET,
        "partition_column": PARTICAO,
        "origem_particao":  "artefato_supervisionado",
        "cohort_kind":      manifest.get("cohort_kind"),
    }
    r.metrica(n_features_processadas=N_FEATURES_PROCESSADAS)

print("Run registrado em:", r.pasta)

Execução registrada em: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\runs\20261107T002441274915Z_06_pipeline_supervisionado
Run registrado em: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\runs\20261107T002441274915Z_06_pipeline_supervisionado
